In [0]:
from pyspark.sql import Row

data = (
    [Row(customer_id=101, amount=100)] * 1000
    + [Row(customer_id=102, amount=200)] * 100
    + [Row(customer_id=103, amount=300)] * 100
    + [Row(customer_id=104, amount=400)] * 100
)

skew_df = spark.createDataFrame(data)

skew_df.groupBy("customer_id").count().show()

In [0]:
skew_partition= skew_df.repartition(4,"customer_id")
skew_partition.show()

In [0]:
from pyspark.sql.functions import spark_partition_id

skew_partition \
    .withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

In [0]:
from pyspark.sql.functions import floor, rand

salted_df = skew_df.withColumn(
    "salt",
    floor(rand(seed=42) * 5)
)

salted_df.show()

In [0]:
salted_partitioned = salted_df.repartition(
    4,
    "customer_id",
    "salt"
)
salted_partitioned \
    .withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()